In [1]:
!apt-get -qq install ffmpeg

!pip install -q openai-whisper
!pip install -q cohere
!pip install -q gTTS
!pip install -q pydub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.


In [3]:
from IPython.display import Javascript, display
from google.colab import output

RECORD = """
async function record(){
const stream = await navigator.mediaDevices.getUserMedia({audio:true});

const recorder = new MediaRecorder(stream);

let chunks=[];

recorder.ondataavailable=e=>chunks.push(e.data);

recorder.start();

await new Promise(r=>setTimeout(r,5000));

recorder.stop();

await new Promise(r=>recorder.onstop=r);

const blob=new Blob(chunks);

const reader=new FileReader();

reader.readAsDataURL(blob);

return await new Promise(r=>{
reader.onloadend=()=>r(reader.result);
});
}
"""

display(Javascript(RECORD))

audio = output.eval_js("record()")

<IPython.core.display.Javascript object>

In [4]:
from base64 import b64decode
from pydub import AudioSegment

audio_data = b64decode(audio.split(",")[1])

with open("recording.webm", "wb") as f:
    f.write(audio_data)

sound = AudioSegment.from_file("recording.webm")
sound.export("audio.wav", format="wav")

print("تم إنشاء audio.wav بنجاح ✅")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


تم إنشاء audio.wav بنجاح ✅


In [5]:
import whisper

# تحميل النموذج (أول مرة قد يستغرق دقيقة أو دقيقتين)
model = whisper.load_model("base")

# تحويل الصوت إلى نص
result = model.transcribe("audio.wav")

print("النص المستخرج:")
print(result["text"])

100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 62.3MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


النص المستخرج:
 How are you?


In [10]:
import cohere

COHERE_API_KEY = "YOUR_COHERE_API_KEY"
co = cohere.Client(COHERE_API_KEY)

response = co.chat(
    message=result["text"]
)

print(response.text)

As an AI language model, I don't have feelings or emotions, so I don't experience well-being in the same way humans do. However, I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?


In [11]:
from gtts import gTTS

tts = gTTS(text=response.text, lang="ar")

tts.save("response.mp3")

print("تم إنشاء الملف response.mp3 بنجاح ✅")

تم إنشاء الملف response.mp3 بنجاح ✅


In [12]:
from IPython.display import Audio

Audio("response.mp3")